In [3]:
import time
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from matplotlib.ticker import PercentFormatter
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
START_TIME = time.time()

# Chart style: thin marks, light grid, no top/right frame.
COLOR_MAIN = "#2a78d6"   # blue   - single-series charts, "satisfied"
COLOR_ALT = "#eb6834"    # orange - "not satisfied"
plt.rcParams.update({
    "figure.dpi": 100,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3df",
    "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "axes.titleweight": "bold",
})

print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("scikit-learn", sklearn.__version__)
print("lightgbm    ", lgb.__version__)

pandas       3.0.6
numpy        2.5.4
scikit-learn 1.9.1
lightgbm     4.7.0


In [4]:
REQUIRED_FILES = ["train.csv", "test.csv", "sample_submission.csv"]


def has_all_files(folder: Path) -> bool:
    return all((folder / name).is_file() for name in REQUIRED_FILES)


def find_data_dir() -> Path:
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        candidates = sorted({p.parent for p in kaggle_root.glob("**/train.csv") if has_all_files(p.parent)})
    else:
        local = [p.resolve() for p in (Path("data"), Path("../data")) if has_all_files(p)]
        candidates = list(dict.fromkeys(local))  # drop duplicates, keep order

    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            f"데이터 폴더를 찾지 못했습니다. {REQUIRED_FILES} 세 파일이 모두 있는 폴더가 필요합니다.\n"
            "- Kaggle: 오른쪽 Input 패널에서 대회 데이터를 추가하세요.\n"
            "- 로컬: 노트북과 같은 위치의 data/ 폴더에 파일을 넣으세요."
        )
    raise RuntimeError(
        "데이터 폴더 후보가 여러 개입니다. 사용할 폴더를 DATA_DIR에 직접 지정하세요:\n"
        + "\n".join(f"  - {c}" for c in candidates)
    )


DATA_DIR = find_data_dir()
print("DATA_DIR =", DATA_DIR)

DATA_DIR = /Users/youngjin/ai/kaggle_practice/Kaggle_Airline_Satisfaction/data
